# 🩺 Prenatal Anemia Screening – Full Training Pipeline

**Run this on Google Colab with GPU enabled.**

Go to: `Runtime → Change runtime type → GPU (T4)`

This notebook will:
1. Clone your GitHub repo
2. Install dependencies
3. Train all 3 models (Image, Tabular, Fusion)
4. Save comprehensive results for the research paper
5. Zip everything for download

## 📦 Step 1 – Setup & Clone Repo

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repo
!rm -rf PrenatalAnemia
!git clone https://github.com/Manideep1617-wq/Prenatal-Anemia.git PrenatalAnemia
%cd PrenatalAnemia
!ls -la

In [ ]:
# Install dependencies
!pip install -q xgboost>=2.0.0 shap opencv-python-headless altair fpdf torchcam seaborn

## 📂 Step 2 – Upload Your Data

You need to upload your `data/` folder. **Two options:**

**Option A – Upload a zip file** (recommended):
1. Zip your entire `data/` folder on your PC
2. Run the cell below and upload the zip

**Option B – Mount Google Drive:**
1. Upload the data folder to your Google Drive
2. Uncomment the Drive mount cell below

In [ ]:
# OPTION A: Upload data.zip
# This will open a file picker — upload your data.zip file

import os
from google.colab import files

print("📁 Upload your data.zip file (containing cbc_dataframe.csv, Dataset - Updated.csv, Dataset_sample/ folder)")
uploaded = files.upload()

# Extract
!mkdir -p data
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        !unzip -o "{fname}" -d data/
        print(f"✅ Extracted {fname}")

# Check what's inside
!echo "\n📂 Contents of data/:"
!find data/ -maxdepth 2 -type f | head -20
!echo "\n📂 Image folders:"
!ls data/Dataset_sample/ 2>/dev/null || echo "⚠️ Dataset_sample not found — check zip structure"

In [ ]:
# # OPTION B: Mount Google Drive (uncomment if using Drive instead of upload)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/PrenatalAnemia/data ./data
# !ls data/

## ⚙️ Step 3 – Fix Config Paths for Colab

In [ ]:
# Update config.yaml to use Colab paths
import yaml

colab_config = {
    'paths': {
        'data_dir': '/content/PrenatalAnemia/data',
        'image_dir': '/content/PrenatalAnemia/data/Dataset_sample',
        'results_dir': '/content/PrenatalAnemia/results',
    },
    'training': {
        'seed': 42,
        'batch_size': 32,
        'image_epochs': 10,
        'lr': 0.0001,
    }
}

with open('src/config.yaml', 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

print("✅ Config updated for Colab:")
!cat src/config.yaml

## 🔍 Step 4 – Verify Data Is Ready

In [ ]:
import pandas as pd
import os

# Check tabular data
cbc_path = '/content/PrenatalAnemia/data/cbc_dataframe.csv'
mat_path = '/content/PrenatalAnemia/data/Dataset - Updated.csv'
img_dir  = '/content/PrenatalAnemia/data/Dataset_sample'

print("=" * 50)
if os.path.exists(cbc_path):
    cbc = pd.read_csv(cbc_path)
    print(f"✅ CBC data: {len(cbc):,} rows, {len(cbc.columns)} columns")
else:
    print("❌ CBC data NOT found")

if os.path.exists(mat_path):
    mat = pd.read_csv(mat_path)
    print(f"✅ Maternal data: {len(mat):,} rows, {len(mat.columns)} columns")
else:
    print("❌ Maternal data NOT found")

if os.path.isdir(img_dir):
    for folder in ['Eye_Anemic', 'Eye_Non_Anemic', 'Nail_Anemic', 'Nail_Non_Anemic']:
        fp = os.path.join(img_dir, folder)
        if os.path.isdir(fp):
            count = len([f for f in os.listdir(fp) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            print(f"   📁 {folder}: {count} images")
else:
    print("❌ Image folder NOT found")
print("=" * 50)

## 🚀 Step 5 – Run Full Training Pipeline

This trains all 3 models and saves everything:
- Phase 1: Image Model (ResNet-18)
- Phase 2: Tabular Model (XGBoost)
- Phase 3: Transformer Fusion Model

In [ ]:
# Run the full training pipeline
!python -m src.train

## 📊 Step 6 – View Results

In [ ]:
# Display the consolidated metrics
import json

with open('results/metrics.json', 'r') as f:
    metrics = json.load(f)

print("=" * 60)
print("  📊 TRAINING RESULTS SUMMARY")
print("=" * 60)
summary = metrics['summary']
print(f"  Image Model   → Acc: {summary['image_accuracy']:.4f}  F1: {summary['image_f1']:.4f}  AUC: {summary['image_auc']}")
print(f"  Tabular Model → Acc: {summary['tabular_accuracy']:.4f}  F1: {summary['tabular_f1']:.4f}  AUC: {summary['tabular_auc']}")
print(f"  Fusion Model  → Acc: {summary['fusion_accuracy']:.4f}  F1: {summary['fusion_f1']:.4f}")
print(f"\n  Training time: {metrics['training_time_minutes']:.1f} minutes")
print(f"  Device: {metrics['device']}")
print("=" * 60)

In [ ]:
# Display all generated figures
from IPython.display import display, Image as IPImage
import glob

fig_files = sorted(glob.glob('results/figures/*.png'))
print(f"📈 {len(fig_files)} figures generated:\n")
for fig_path in fig_files:
    print(f"--- {os.path.basename(fig_path)} ---")
    display(IPImage(filename=fig_path, width=600))
    print()

In [ ]:
# Show classification reports
for model_name in ['image', 'tabular', 'fusion']:
    report_path = f'results/records/{model_name}_classification_report.txt'
    if os.path.exists(report_path):
        print(f"\n{'='*50}")
        print(f"  {model_name.upper()} MODEL – Classification Report")
        print(f"{'='*50}")
        with open(report_path) as f:
            print(f.read())

In [ ]:
# Show feature importance (tabular model)
fi_path = 'results/records/tabular_feature_importance.csv'
if os.path.exists(fi_path):
    fi = pd.read_csv(fi_path)
    print("\n🔬 XGBoost Feature Importance:")
    print(fi.to_string(index=False))

## 📥 Step 7 – Download All Results

This zips the entire `results/` folder for you to download and bring back.

In [ ]:
# Zip all results for download
!zip -r /content/prenatal_anemia_results.zip results/

from google.colab import files
files.download('/content/prenatal_anemia_results.zip')

print("\n✅ Download started! This zip contains:")
print("   📁 results/checkpoints/   → Model weights")
print("   📁 results/records/        → Epoch logs, classification reports, confusion matrices")
print("   📁 results/figures/        → All plots (PNG) for the research paper")
print("   📄 results/metrics.json    → Consolidated metrics with hyperparams & dataset stats")

## 🔁 Step 8 – List All Generated Files

In [ ]:
# Show everything that was generated
import os

print("\n📂 Complete results directory:")
for root, dirs, fnames in os.walk('results/'):
    level = root.replace('results/', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for fn in sorted(fnames):
        fpath = os.path.join(root, fn)
        size = os.path.getsize(fpath)
        if size > 1024*1024:
            size_str = f"{size/1024/1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"{sub_indent}📄 {fn}  ({size_str})")